<a href="https://colab.research.google.com/github/deetijasmitha/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deetijasmitha/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
# Connect your GitHub repository to this Colab runtime

import os

!git clone https://github.com/deetijasmitha/flyrank-ml-internship.git

%cd flyrank-ml-internship

print("Repository connected successfully.")
print("Current directory:", os.getcwd())

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 179, done.
remote: Counting objects: 100% (179/179), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 179 (delta 80), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (179/179), 1.91 MiB | 14.52 MiB/s, done.
Resolving deltas: 100% (80/80), done.
/content/flyrank-ml-internship
Repository connected successfully.
Current directory: /content/flyrank-ml-internship


In [7]:
# Check the repository structure

print("work folder exists:", os.path.exists("work"))
print("work/notebooks exists:", os.path.exists("work/notebooks"))
print("work/outputs exists:", os.path.exists("work/outputs"))
print("outputs folder exists:", os.path.exists("outputs"))

print("\nFiles in work:")
print(os.listdir("work"))

print("\nFiles in outputs:")
print(os.listdir("outputs"))

work folder exists: True
work/notebooks exists: True
work/outputs exists: False
outputs folder exists: True

Files in work:
['capstone_report_template.md', 'notebooks', 'README.md']

Files in outputs:
['model_report.md', 'charts', 'refresh_queue_sample.csv']


In [8]:
# Find the baseline output and the code that creates it

import os

repo_root = "/content/flyrank-ml-internship"

print("Searching for baseline_action_score.csv...\n")

for root, dirs, files in os.walk(repo_root):
    for file in files:
        if "baseline_action" in file.lower():
            print(os.path.join(root, file))

print("\nSearching W04 notebook for baseline_action_score references...\n")

w04_path = os.path.join(
    repo_root,
    "work",
    "notebooks",
    "w04_baseline_score.ipynb"
)

with open(w04_path, "r", encoding="utf-8") as f:
    w04_text = f.read()

for line in w04_text.splitlines():
    if "baseline_action_score" in line:
        print(line[:500])

Searching for baseline_action_score.csv...


Searching W04 notebook for baseline_action_score references...

        "*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*"
            "Output: work/outputs/baseline_action_score.csv\n",
        "output_path = \"work/outputs/baseline_action_score.csv\"\n",
        "    \"work/outputs/baseline_action_score.csv\"\n",
        "- [x] The ranked queue is written to `work/outputs/baseline_action_score.csv`\n",


In [9]:
# ML-10 preparation — recreate the W04 baseline output

import os
import duckdb
import pandas as pd

# Create DuckDB connection
con = duckdb.connect()

print("DuckDB connection created.")

# Check whether the Hugging Face secret is available
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN is not available in Colab Secrets. "
        "Please make sure your HF_TOKEN is configured."
    )

print("HF_TOKEN loaded successfully.")

# Configure Hugging Face access
try:
    con.execute("DROP SECRET hf_token")
except:
    pass

con.execute("""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("Hugging Face token configured.")

# Load the March 2026 data used by W04
march_query = """
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
    SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
"""

score_df = con.sql(march_query).df()

print("March content-client rows:", len(score_df))

DuckDB connection created.
HF_TOKEN loaded successfully.
Hugging Face token configured.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March content-client rows: 176738


In [10]:
# Reset to the cloned FlyRank repository

import os

REPO_PATH = "/content/flyrank-ml-internship"

if not os.path.exists(REPO_PATH):
    raise FileNotFoundError(
        "The FlyRank repository is not cloned in this Colab runtime."
    )

os.chdir(REPO_PATH)

print("Connected repository:")
print(os.getcwd())

print("\nW04 notebook exists:")
print(
    os.path.exists(
        "work/notebooks/w04_baseline_score.ipynb"
    )
)

print("\nW07 notebook exists:")
print(
    os.path.exists(
        "work/notebooks/w07_action_playbook.ipynb"
    )
)


Connected repository:
/content/flyrank-ml-internship

W04 notebook exists:
True

W07 notebook exists:
True


In [13]:
# ML-10 — Section 1 preparation
# Recreate the W04 baseline output in the current Colab runtime

import os
import duckdb
import pandas as pd

# ---------------------------------------------------------
# 1. Create DuckDB connection
# ---------------------------------------------------------

con = duckdb.connect()

print("DuckDB connection created.")

# ---------------------------------------------------------
# 2. Configure Hugging Face
# ---------------------------------------------------------

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found in Colab Secrets."
    )

try:
    con.execute("DROP SECRET hf_token")
except:
    pass

con.execute(
    """
    CREATE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN ?
    )
    """,
    [HF_TOKEN]
)

print("HF_TOKEN configured successfully.")

# ---------------------------------------------------------
# 3. Load March 2026 content performance
# ---------------------------------------------------------

march_df = con.sql(
    """
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
        SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()

print("March content-client rows:", len(march_df))

# ---------------------------------------------------------
# 4. Apply W04 baseline scoring
# ---------------------------------------------------------

march_df["gsc_clicks"] = march_df["gsc_clicks"].fillna(0)
march_df["gsc_avg_position"] = march_df["gsc_avg_position"].fillna(999)

# Click score
march_df["click_score"] = 0

march_df.loc[
    march_df["gsc_clicks"] <= 2,
    "click_score"
] = 2

march_df.loc[
    (march_df["gsc_clicks"] > 2) &
    (march_df["gsc_clicks"] <= 10),
    "click_score"
] = 1

# Position score
march_df["position_score"] = 0

march_df.loc[
    march_df["gsc_avg_position"] > 5,
    "position_score"
] = 1

march_df.loc[
    march_df["gsc_avg_position"] > 10,
    "position_score"
] = 2

march_df.loc[
    march_df["gsc_avg_position"] > 20,
    "position_score"
] = 3

# Combined score
march_df["score"] = (
    march_df["click_score"] +
    march_df["position_score"]
)

# W04 reason/action
march_df["reason_code"] = "LOW_SEARCH_PERFORMANCE"
march_df["action"] = "REVIEW_REFRESH"

# ---------------------------------------------------------
# 5. Rank
# ---------------------------------------------------------

march_df = march_df.sort_values(
    by=[
        "score",
        "gsc_clicks",
        "gsc_avg_position"
    ],
    ascending=[
        False,
        True,
        False
    ]
).reset_index(drop=True)

march_df["rank"] = range(
    1,
    len(march_df) + 1
)

# ---------------------------------------------------------
# 6. Create final queue
# ---------------------------------------------------------

baseline_queue = march_df[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action"
    ]
].copy()

# ---------------------------------------------------------
# 7. Save the required file
# ---------------------------------------------------------

os.makedirs("work/outputs", exist_ok=True)

baseline_path = "work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(
    baseline_path,
    index=False
)

# ---------------------------------------------------------
# 8. Self-check
# ---------------------------------------------------------

print("\n========================================")
print("W04 BASELINE OUTPUT CHECK")
print("========================================")

print("Rows:", len(baseline_queue))
print("Columns:", list(baseline_queue.columns))
print("Duplicate rows:", baseline_queue.duplicated().sum())
print("File exists:", os.path.exists(baseline_path))
print("Output:", baseline_path)

print("\nScore distribution:")
display(
    baseline_queue["score"]
    .value_counts()
    .sort_index()
)

print("\nTop 10:")
display(
    baseline_queue.head(10)
)

DuckDB connection created.
HF_TOKEN configured successfully.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March content-client rows: 176738

W04 BASELINE OUTPUT CHECK
Rows: 176738
Columns: ['rank', 'client_hash_id', 'content_hash_id', 'score', 'reason_code', 'action']
Duplicate rows: 0
File exists: True
Output: work/outputs/baseline_action_score.csv

Score distribution:


,count
score,
0,7795
1,12391
2,38691
3,48545
4,29159
5,40157



Top 10:


,rank,client_hash_id,content_hash_id,score,reason_code,action
0,1,client_08a6a72ff48e62c0,content_9e8c3b83214c180d,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
1,2,client_3ffa76342f366962,content_06589faf15cc8488,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
2,3,client_3ffa76342f366962,content_36cc2bda86ee726a,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
3,4,client_08a6a72ff48e62c0,content_11187e07e5ee9f43,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
4,5,client_3ffa76342f366962,content_efce4eda2b012964,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
5,6,client_3ffa76342f366962,content_0cec599cfeab8b7f,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
6,7,client_23a62021009f63c4,content_61b375eafb1d4c27,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
7,8,client_f623b01661d4bfe4,content_fc468c5940d16ea3,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
8,9,client_3ffa76342f366962,content_3758dd311e8033f7,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH
9,10,client_3ffa76342f366962,content_d1b44ca865290810,5,LOW_SEARCH_PERFORMANCE,REVIEW_REFRESH


## 1. Ranked actions + reason codes


The action playbook converts model or performance signals into a ranked queue for human review.

Higher-priority items are surfaced first based on their measured score and supporting performance signals. Each item receives a reason code so that the recommended action is explainable rather than being based on a score alone.

The queue is intended for prioritization and decision-support. A high-ranked item does not mean that a content change should be made automatically. Human review is required before any refresh, rewrite, consolidation, or other content action.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.